# Landslide Random Forest + SMOTE — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `landslide_data.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn imbalanced-learn joblib

import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_FILE = "landslide_data.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)

FEATURES = ["pressure", "soil_moisture"]
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2, "Critical": 2}
CLASS_NAMES = ["Low", "Medium", "High"]

df["target"] = df["risk_level"].map(LABEL_MAP)
X = df[FEATURES].copy()
y = df["target"].astype(int).copy()

print("Rows:", len(df))
print("Overall class counts:", Counter(y))
print("Missing feature values:", X.isna().sum().to_dict())

## Leakage-free tabular pipeline

The real held-out test set is isolated **before** imputation, scaling, and SMOTE.

Processing order:
1. stratified 80/20 train-test split;
2. fit imputer on training only;
3. fit scaler on training only;
4. apply SMOTE only to transformed training data;
5. train Random Forest;
6. evaluate once on untouched real test samples.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp  = imputer.transform(X_test_raw)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print("Before SMOTE:", Counter(y_train))
smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print("After train-only SMOTE:", Counter(y_train_bal))
print("Untouched test counts:", Counter(y_test))

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1
)
rf_model.fit(X_train_bal, y_train_bal)

y_pred = rf_model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy       : {acc:.6f}")
print(f"Macro precision: {prec:.6f}")
print(f"Macro recall   : {rec:.6f}")
print(f"Macro F1       : {f1:.6f}")
print()
print(classification_report(
    y_test, y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title("Landslide Random Forest — Leakage-Free Test Set")
plt.show()

pd.DataFrame({
    "true_class": [CLASS_NAMES[i] for i in y_test.to_numpy()],
    "predicted_class": [CLASS_NAMES[i] for i in y_pred]
}).to_csv("landslide_rf_leakage_free_test_predictions.csv", index=False)

In [ ]:
tree_depths = np.array([est.tree_.max_depth for est in rf_model.estimators_])
print("Number of trees:", len(rf_model.estimators_))
print("Mean tree depth :", tree_depths.mean())
print("Max tree depth  :", tree_depths.max())

In [ ]:
bundle = {
    "features": FEATURES,
    "class_names": CLASS_NAMES,
    "label_map": LABEL_MAP,
    "imputer": imputer,
    "scaler": scaler,
    "model": rf_model,
    "split": {"train": 0.80, "test": 0.20},
    "smote": "training data only",
    "seed": SEED
}
joblib.dump(bundle, "landslide_rf_leakage_free_bundle.joblib")
print("Saved landslide_rf_leakage_free_bundle.joblib")